# ST-Reason · 01b STAR set (week 2)

STAR replaces the self-recorded clips. This notebook builds everything the gap test, the oracle runs and the detection metrics need, and saves it to Drive:

| Step | Output on Drive |
| --- | --- |
| 1. Download STAR val, class files and Charades annotations | `data/star/raw/` |
| 2. Check the data format (stops with a clear message if it differs from what the code expects) | printed |
| 3. Find each sequence question's anchor and answer actions, with exact times | `data/star/questions_star_val.csv` |
| 4. Sample the STAR sets (frozen once saved) | `data/subsets/star_main.csv`, `star_gap.csv` |
| 5. Cut the clips, then make pause and sham versions | `data/videos_star_eval.zip`, `data/star/clip_manifest.csv` |
| 6. Ground-truth boxes for detection metrics | `data/star/boxes/` |
| 7. Oracle memory inputs (ground-truth events, objects, relations) | `data/star/oracle/` |

**Runtime:** CPU is enough. Step 5 needs about 15 GB free on the Colab disk for the Charades video zip; the free runtime has plenty, and nothing large is stored on Drive.

**If a download fails** (some networks block these servers): download the file in your browser from the link printed, upload it to `MyDrive/ST-Reason/data/source/star/`, and re-run. Files found there are always used first.

In [1]:
# --- 0. Bootstrap ------------------------------------------------------------------------------
import os, glob
ST_NOTEBOOK = '01b_star'
ST_QUIET = True
if 'COLAB_RELEASE_TAG' in os.environ:
    from google.colab import drive
    drive.mount('/content/drive')
    SETUP_NB = '/content/drive/MyDrive/ST-Reason/notebooks/00_setup_and_utils.ipynb'
elif os.path.exists('/kaggle/working'):
    SETUP_NB = (glob.glob('/kaggle/input/**/00_setup_and_utils.ipynb', recursive=True) or ['00_setup_and_utils.ipynb'])[0]
else:
    SETUP_NB = os.environ.get('ST_SETUP_NB', '00_setup_and_utils.ipynb')
%run -i $SETUP_NB

import pandas as pd, numpy as np, pickle, urllib.request
set_seed()
RUN = Run(params={'step': 'star_set'})
STAR = P['data'] / 'star'
for sub in ['raw', 'boxes', 'oracle']:
    (STAR / sub).mkdir(parents=True, exist_ok=True)
MANUAL = P['source'] / 'star'          # optional hand-uploaded copies of any download
MANUAL.mkdir(parents=True, exist_ok=True)
STAR_LOCAL = P['local_tmp'] / 'star'
STAR_LOCAL.mkdir(parents=True, exist_ok=True)

Mounted at /content/drive
Runtime: colab | project folder: /content/drive/MyDrive/ST-Reason | local scratch: /content/st_local
config hash: 2a9e0e4f87
[09:31:38] INFO  run started: 01b_star-20261016-093129-4239b5


## 1. Downloads

STAR files come from the STAR authors' public storage, the class files from their GitHub repository, and the Charades annotations and videos from AI2. Every URL is in `config.yaml` under `star`, so a moved link can be fixed without editing code.

In [2]:
S3 = 'https://star-benchmark.s3.us-east.cloud-object-storage.appdomain.cloud'
STAR_DEFAULTS = {
    'urls': {
        'STAR_val.json': f'{S3}/Question_Answer_SituationGraph/STAR_val.json',
        'Video_Segments.csv': f'{S3}/Situation_Video_Data/Video_Segments.csv',
        'object_bbox_and_relationship.pkl': f'{S3}/Annotations/object_bbox_and_relationship.pkl',
        'person_bbox.pkl': f'{S3}/Annotations/person_bbox.pkl',
        'Charades.zip': 'https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip',
        'Charades_v1_480.zip': 'https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip',
    },
    'alt_urls': {
        'Charades.zip': 'http://ai2-website.s3.amazonaws.com/data/Charades.zip',
        'Charades_v1_480.zip': 'http://ai2-website.s3.amazonaws.com/data/Charades_v1_480.zip',
    },
    'code_repo': 'https://github.com/csbobby/STAR_Benchmark.git',
    'n_sequence': 240, 'n_interaction': 120, 'n_gap': 120, 'max_q_per_video': 2,
    'pause_medium_s': 10.0, 'pause_long_s': 25.0, 'sham_margin_s': 0.5,
}
if 'star' not in CFG:
    CFG['star'] = STAR_DEFAULTS
    with open(CONFIG_PATH, 'w') as f:
        yaml.safe_dump(CFG, f, sort_keys=False)
SC = _deep_merge(STAR_DEFAULTS, CFG['star'])

def fetch(name, required=True):
    """Return a local path for `name`: hand-uploaded copy first, then cached download, then the web."""
    for p in [MANUAL / name, STAR_LOCAL / name]:
        if p.exists() and p.stat().st_size > 0:
            return p
    out = STAR_LOCAL / name
    for url in [SC['urls'].get(name), SC['alt_urls'].get(name)]:
        if not url:
            continue
        try:
            log(f'downloading {name} from {url}')
            subprocess.run(['wget', '-q', '-c', '-O', str(out), url], check=True, timeout=6 * 3600)
            if out.exists() and out.stat().st_size > 0:
                log(f'{name}: {_gb(out.stat().st_size)} GB')
                return out
        except Exception as e:
            log(f'{name}: download failed from {url} ({e.__class__.__name__})', 'WARN')
            out.unlink(missing_ok=True)
    msg = f'{name} not available. Download it from {SC["urls"].get(name)} and upload it to {MANUAL}, then re-run.'
    if required:
        raise FileNotFoundError(msg)
    log(msg, 'WARN')
    return None

star_val_path = fetch('STAR_val.json')
segments_path = fetch('Video_Segments.csv', required=False)

repo = P['local_repos'] / 'STAR_Benchmark'
if not (repo / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', '-q', SC['code_repo'], str(repo)], check=True)
CLS = repo / 'annotations' / 'STAR_classes'
for f in ['action_classes.txt', 'action_mapping.txt', 'verb_classes.txt', 'object_classes.txt',
          'relationship_classes.txt', 'video_fps']:
    shutil.copy2(CLS / f, STAR / 'raw' / f)

charades_zip = fetch('Charades.zip')
with zipfile.ZipFile(charades_zip) as zf:
    for n in zf.namelist():
        base = Path(n).name
        if base in ('Charades_v1_train.csv', 'Charades_v1_test.csv', 'Charades_v1_classes.txt'):
            (STAR / 'raw' / base).write_bytes(zf.read(n))
log('STAR val, class files and Charades annotations ready')

[09:31:39] INFO  downloading STAR_val.json from https://star-benchmark.s3.us-east.cloud-object-storage.appdomain.cloud/Question_Answer_SituationGraph/STAR_val.json
[09:31:40] INFO  STAR_val.json: 0.06 GB
[09:31:40] INFO  downloading Video_Segments.csv from https://star-benchmark.s3.us-east.cloud-object-storage.appdomain.cloud/Situation_Video_Data/Video_Segments.csv
[09:31:41] INFO  Video_Segments.csv: 0.0 GB
[09:31:43] INFO  downloading Charades.zip from https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades.zip
[09:31:44] INFO  Charades.zip: 0.0 GB
[09:31:45] INFO  STAR val, class files and Charades annotations ready


## 2. Check the data format

The code relies on these STAR fields: `question_id`, `question`, `video_id`, `start`, `end`, `answer`, `choices` (each with `choice` and `choice_program`), `question_program` and `situations` (per keyframe: `actions`, `bbox_labels`, `rel_pairs`, `rel_labels`). If any is missing, this cell stops and prints one full record so the code can be adjusted.

In [3]:
with open(star_val_path) as f:
    SV = json.load(f)
REQUIRED = ['question_id', 'question', 'video_id', 'start', 'end', 'answer', 'choices', 'question_program', 'situations']
missing = [k for k in REQUIRED if k not in SV[0]]
sit0 = next(iter(SV[0]['situations'].values())) if SV[0].get('situations') else {}
missing += [f'situations.{k}' for k in ['actions', 'bbox_labels', 'rel_pairs', 'rel_labels'] if k not in sit0]
missing += [f'choices.{k}' for k in ['choice', 'choice_program'] if SV[0].get('choices') and k not in SV[0]['choices'][0]]
if missing:
    print(json.dumps(SV[0], indent=1, default=str)[:4000])
    raise KeyError(f'STAR format differs from what this notebook expects; missing {missing}. Send this output for a fix.')

qtypes = pd.Series([q['question_id'].split('_')[0] for q in SV]).value_counts().to_dict()
print(f'STAR val: {len(SV)} questions, {len({q["video_id"] for q in SV})} source videos; types {qtypes}')
print('answers present:', all(q.get('answer') not in (None, '') for q in SV[:200]))
print('example:', SV[0]['question'], '->', SV[0]['answer'])
print('example question program:', [p['function'] for p in SV[0]['question_program']])

STAR val: 7098 questions, 914 source videos; types {'Sequence': 3586, 'Interaction': 2398, 'Prediction': 624, 'Feasibility': 490}
answers present: True
example: Which object was tidied up by the person? -> The clothes.
example question program: ['Situations', 'Actions', 'Filter_Actions_with_Verb', 'Unique', 'Query_Objs']


## 3. Anchor and answer actions, with exact times

**Mapping STAR actions to Charades classes.** STAR's action names are Charades' class names with the verb in its base form ("hold some clothes" is Charades' "Holding some clothes"), so they are matched exactly after converting Charades' verbs to their base form. A fuzzy match is used only for anything left over, and is reported.

**Finding the two actions.** STAR's six sequence templates name the actions differently. Some answers are a whole action ("Put down the dish."); others are only an object ("Which object did the person take…") or only a verb. So the notebook does not rely on the answer alone:
- **Anchor:** the action in the clip whose verb and object both appear in the question program.
- **Answer:** the nearest action after (or before) the anchor, in the clip's own order, that fits every verb and object the correct choice names, plus any verb or object the question names besides the anchor.

**Times** come from Charades' exact action annotations, with the first and last STAR keyframe showing the action as a cross-check and fallback. They are converted to seconds within the clip.

**Two gap measures.** Actions in Charades often overlap ("holding a phone" spans "putting the phone down"), so two are kept:
- `gap_s`: answer start minus anchor end (end to start). Negative means the actions overlap.
- `onset_gap_s`: the time between the two actions' starts. It is positive whenever the order matches the question, and it is where pauses are placed.

In [4]:
def read_classes(path):
    out = {}
    for line in open(path):
        parts = line.strip().split(' ', 1)
        if len(parts) == 2:
            out[parts[0]] = parts[1]
    return out

ACTION_NAME = read_classes(STAR / 'raw' / 'action_classes.txt')
VERB = read_classes(STAR / 'raw' / 'verb_classes.txt')
OBJ = read_classes(STAR / 'raw' / 'object_classes.txt')
REL = read_classes(STAR / 'raw' / 'relationship_classes.txt')
AMAP = {a: tuple(v.split()) for a, v in read_classes(STAR / 'raw' / 'action_mapping.txt').items()}
ACTION_VO = {a: (VERB[v], OBJ[o]) for a, (v, o) in AMAP.items()}          # 'a002' -> ('take', 'clothes')
VO_ACTION = {vo: a for a, vo in ACTION_VO.items()}
FPS = pickle.load(open(STAR / 'raw' / 'video_fps', 'rb'))                 # 'BE43L.mp4' -> 29.97
VERB_WORDS, OBJ_WORDS = set(VERB.values()), set(OBJ.values())

# Charades class names -> STAR action ids. STAR names are Charades names with base-form verbs,
# so the match is exact after lemmatising; a fuzzy match handles anything left over.
import itertools
IRREG = {'putting': 'put', 'sitting': 'sit', 'lying': 'lie', 'taking': 'take', 'closing': 'close',
         'tidying': 'tidy', 'throwing': 'throw', 'holding': 'hold', 'washing': 'wash', 'opening': 'open',
         'standing': 'stand', 'eating': 'eat', 'drinking': 'drink', 'watching': 'watch', 'reading': 'read',
         'looking': 'look', 'reaching': 'reach', 'grabbing': 'grab', 'walking': 'walk', 'turning': 'turn',
         'grasping': 'grasp', 'consuming': 'consume'}
def lemma_word(w):
    if w in IRREG:
        return [IRREG[w]]
    if w.endswith('ing') and len(w) > 5:
        b = w[:-3]; out = [b, b + 'e']
        if len(b) > 2 and b[-1] == b[-2]:
            out.append(b[:-1])
        return out
    return [w]
def lemma_variants(name):
    variants = ['']
    for tok in name.lower().split():
        forms = ['/'.join(p) for p in itertools.product(*[lemma_word(x) for x in tok.split('/')])]
        variants = [f'{v} {f}'.strip() for v in variants for f in forms][:256]
    return variants
def norm(s):
    return re.sub(r'\s+', ' ', str(s).lower().strip())
def stems(s):
    out = set()
    for w in re.findall(r'[a-z]+', s.lower()):
        for suf in ('ing', 'es', 's', 'ed'):
            if w.endswith(suf) and len(w) > len(suf) + 2:
                w = w[: -len(suf)]; break
        out.add(w)
    return out - {'some', 'something', 'somewhere', 'a', 'the', 'from', 'on', 'in', 'into', 'with', 'of', 'at', 'to'}

CHA_CLASSES = read_classes(STAR / 'raw' / 'Charades_v1_classes.txt')
STAR_BY_NAME = {norm(n): a for a, n in ACTION_NAME.items()}
STAR2CHA, how = {}, {}
for c, cname in CHA_CLASSES.items():
    for v in lemma_variants(cname):
        if v in STAR_BY_NAME and STAR_BY_NAME[v] not in STAR2CHA:
            STAR2CHA[STAR_BY_NAME[v]] = c; how[STAR_BY_NAME[v]] = 'exact'; break
for a, name in ACTION_NAME.items():
    if a in STAR2CHA:
        continue
    best, score = None, 0.0
    for c, cname in CHA_CLASSES.items():
        sa, sc = stems(name), stems(cname); j = len(sa & sc) / max(1, len(sa | sc))
        if j > score:
            best, score = c, j
    if score >= 0.5:
        STAR2CHA[a] = best; how[a] = 'fuzzy'
n_exact = sum(v == 'exact' for v in how.values())
print(f'STAR actions mapped to Charades classes: {len(STAR2CHA)}/{len(ACTION_NAME)} '
      f'({n_exact} exact, {len(STAR2CHA) - n_exact} fuzzy)')
unmapped = [ACTION_NAME[a] for a in ACTION_NAME if a not in STAR2CHA]
if unmapped:
    print('  unmapped (times fall back to keyframes):', unmapped)

CHA_ACTIONS = {}                                                        # video -> list of (class, start, end)
for split in ['train', 'test']:
    df = pd.read_csv(STAR / 'raw' / f'Charades_v1_{split}.csv', dtype=str)
    for vid, acts in zip(df['id'], df['actions'].fillna('')):
        CHA_ACTIONS[vid] = [(c, float(s), float(e)) for c, s, e in
                            (a.split() for a in acts.split(';') if len(a.split()) == 3)]

def program_words(prog):
    return [w for p in prog for w in p.get('value_input', []) if isinstance(w, str)]

def keyframe_span(q, action):
    fps = FPS.get(q['video_id'] + '.mp4')
    frames = sorted(int(f) for f, s in q['situations'].items() if action in s.get('actions', []))
    if not frames or not fps:
        return None
    return frames[0] / fps, frames[-1] / fps

def charades_span(q, action, around):
    c = STAR2CHA.get(action)
    cands = [(s, e) for cc, s, e in CHA_ACTIONS.get(q['video_id'], []) if cc == c]
    if not cands:
        return None
    if around is None:
        return min(cands)
    mid = (around[0] + around[1]) / 2
    return min(cands, key=lambda se: 0 if se[0] <= mid <= se[1] else min(abs(se[0] - mid), abs(se[1] - mid)))

def clip_actions_in_order(q):
    """STAR action ids in the clip, ordered by the first keyframe that shows them."""
    order = []
    for f in sorted(q['situations'], key=int):
        for a in q['situations'][f].get('actions', []):
            if a in ACTION_VO and a not in order:
                order.append(a)
    return order

def without_one(words, w):
    out = list(words)
    if w in out:
        out.remove(w)
    return out

def resolve(q):
    """Describe the anchor and answer actions and the gaps, or give the reason a question was skipped."""
    qtype = q['question_id'].split('_')[0]
    rec = {'uid': 'star_' + q['question_id'], 'question_id': q['question_id'], 'type': qtype,
           'template': q['question_id'].split('_')[1] if '_' in q['question_id'] else '',
           'video_id': q['video_id'], 'clip_start': float(q['start']), 'clip_end': float(q['end']),
           'question': q['question'], 'answer_text': q['answer']}
    opts = [c['choice'] for c in q['choices']]
    rec.update({f'a{i}': o for i, o in enumerate(opts)})
    rec['answer_idx'] = opts.index(q['answer']) if q['answer'] in opts else -1
    rec['n_options'] = len(opts)
    if qtype != 'Sequence':
        rec['status'] = 'not_sequence'
        return rec
    funcs = [p['function'] for p in q['question_program']]
    direction = 'after' if 'Filter_After_Actions' in funcs else ('before' if 'Filter_Before_Actions' in funcs else None)
    qw = program_words(q['question_program'])
    qv, qo = [w for w in qw if w in VERB_WORDS], [w for w in qw if w in OBJ_WORDS]
    correct = next((c for c in q['choices'] if c['choice'] == q['answer']), None)
    cw = program_words(correct['choice_program']) if correct else []
    cv, co = [w for w in cw if w in VERB_WORDS], [w for w in cw if w in OBJ_WORDS]
    rec['direction'] = direction
    if not direction:
        rec['status'] = 'no_direction'
        return rec
    acts = clip_actions_in_order(q)
    anchors = [a for a in acts if ACTION_VO[a][0] in qv and ACTION_VO[a][1] in qo]
    if len(anchors) > 1 and qv and qo:                        # prefer the first verb/object pair the program names
        pref = [a for a in anchors if ACTION_VO[a] == (qv[0], qo[0])]
        anchors = pref or anchors
    if not anchors:
        rec['status'] = 'anchor_not_found'
        return rec
    anchor = anchors[0]
    qv_rest, qo_rest = without_one(qv, ACTION_VO[anchor][0]), without_one(qo, ACTION_VO[anchor][1])
    def fits(a):
        v, o = ACTION_VO[a]
        v_ok = (v in cv) if cv else ((v in qv_rest) if qv_rest else True)
        o_ok = (o in co) if co else ((o in qo_rest) if qo_rest else True)
        return v_ok and o_ok
    i = acts.index(anchor)
    side = acts[i + 1:] if direction == 'after' else acts[:i][::-1]          # nearest first
    cands = [a for a in side if a != anchor and fits(a)]
    rec['answer_match'] = 'choice' if (cv or co) else ('question' if (qv_rest or qo_rest) else 'order_only')
    if not cands:
        rec['status'] = 'answer_not_found'
        return rec
    answer = cands[0]
    rec.update({'anchor_action': anchor, 'answer_action': answer,
                'anchor_name': ACTION_NAME.get(anchor), 'answer_name': ACTION_NAME.get(answer)})
    for role, a in [('anchor', anchor), ('answer', answer)]:
        kf = keyframe_span(q, a)
        ch = charades_span(q, a, kf)
        span, src = (ch, 'charades') if ch else (kf, 'keyframes')
        if span is None:
            rec['status'] = f'no_time_{role}'
            return rec
        s, e = span[0] - rec['clip_start'], span[1] - rec['clip_start']
        rec.update({f'{role}_start': round(s, 2), f'{role}_end': round(e, 2), f'{role}_time_source': src,
                    f'{role}_kf_start': round(kf[0] - rec['clip_start'], 2) if kf else None})
    first, second = ('anchor', 'answer') if direction == 'after' else ('answer', 'anchor')
    rec['first_role'], rec['second_role'] = first, second
    rec['gap_s'] = round(rec[f'{second}_start'] - rec[f'{first}_end'], 2)             # end to start
    rec['onset_gap_s'] = round(rec[f'{second}_start'] - rec[f'{first}_start'], 2)     # start to start
    dur = rec['clip_end'] - rec['clip_start']
    inside = all(-0.5 <= rec[f'{r}_start'] and rec[f'{r}_end'] <= dur + 0.5 for r in ('anchor', 'answer'))
    if not inside:
        rec['status'] = 'outside_clip'
    elif rec['onset_gap_s'] <= 0:
        rec['status'] = 'order_conflict'          # annotated times disagree with the question's order
    else:
        rec['status'] = 'ok'
    return rec

SQ = pd.DataFrame([resolve(q) for q in SV])
SQ.to_csv(STAR / 'questions_star_val.csv', index=False)
seq = SQ[SQ.type == 'Sequence']
print('sequence questions by status:', seq.status.value_counts().to_dict())
print('ok by template:', seq[seq.status == 'ok'].template.value_counts().sort_index().to_dict(),
      '| all by template:', seq.template.value_counts().sort_index().to_dict())
ok = seq[seq.status == 'ok']
if len(ok):
    print('answer matched via:', ok.answer_match.value_counts().to_dict())
    print('time source (anchor):', ok.anchor_time_source.value_counts().to_dict())
    print('end-to-start gap (s):', ok.gap_s.describe()[['min', '25%', '50%', '75%', 'max']].round(2).to_dict())
    print('onset gap (s):       ', ok.onset_gap_s.describe()[['min', '25%', '50%', '75%', 'max']].round(2).to_dict())
    print(f'actions overlap in {(ok.gap_s < 0).mean():.0%} of resolved questions')
    both = ok.dropna(subset=['anchor_kf_start'])
    both = both[both.anchor_time_source == 'charades']
    if len(both):
        d = (both.anchor_start - both.anchor_kf_start).abs()
        print(f'Charades vs keyframe start agreement: median |diff| {d.median():.2f}s, within 1 s for {(d <= 1).mean():.0%}')

STAR actions mapped to Charades classes: 111/111 (111 exact, 0 fuzzy)
sequence questions by status: {'ok': 2823, 'outside_clip': 760, 'order_conflict': 3}
ok by template: {'T1': 516, 'T2': 605, 'T3': 442, 'T4': 526, 'T5': 341, 'T6': 393} | all by template: {'T1': 800, 'T2': 614, 'T3': 705, 'T4': 533, 'T5': 536, 'T6': 398}
answer matched via: {'choice': 2823}
time source (anchor): {'charades': 2823}
end-to-start gap (s): {'min': -11.2, '25%': -4.0, '50%': -2.2, '75%': 1.8, 'max': 40.1}
onset gap (s):        {'min': 0.06, '25%': 2.3, '50%': 5.5, '75%': 10.9, 'max': 51.1}
actions overlap in 67% of resolved questions
Charades vs keyframe start agreement: median |diff| 0.53s, within 1 s for 80%


## 4. STAR evaluation sets (frozen once saved)

- **`star_main`**: 240 sequence questions with resolved times, plus 120 interaction questions; at most 2 questions per source video. Used for H1 replication, natural-gap analysis (H2) and the oracle runs. Questions whose answer could only be identified by position (`answer_match = order_only`) are not used.
- **`star_gap`**: 120 of those sequence questions. The pause goes midway between the two actions' **starts**, so it lengthens the time between them even when the actions overlap. The sham pause, of the same length, goes either before the first action starts or after the second one starts (whichever has more room), so it leaves the time between the two starts unchanged. STAR clips are cut tightly around the actions, so the sham usually falls inside the second action. The real pause can also fall inside an action, so the only difference between the two is whether the pause lies between the two starts.

The sets carry a version number. Sets saved by an older version of this notebook (version 1 had the parsing bug fixed above) are re-sampled automatically. That is legitimate only because no results have been produced from them yet.

In [5]:
def take_capped(df, n, cap, seed):
    df = df.sample(frac=1, random_state=seed)
    df = df.assign(_k=df.groupby('video_id').cumcount())
    return df[df._k < cap].drop(columns='_k').head(n).reset_index(drop=True)

STAR_SET_VERSION = 3   # v2 -> v3: sham slot may follow the second action's start (v2 left star_gap empty)
paths = {n: P['subsets'] / f'{n}.csv' for n in ['star_main', 'star_gap']}
def current_version(p):
    try:
        d = pd.read_csv(p, nrows=1)
        return int(d.set_version.iloc[0]) if 'set_version' in d else 1
    except Exception:
        return 0
if all(p.exists() for p in paths.values()) and all(current_version(p) == STAR_SET_VERSION for p in paths.values()):
    STAR_SETS = {n: pd.read_csv(p, dtype={'video_id': str}) for n, p in paths.items()}
    log('STAR sets already frozen: loaded (not re-sampled)')
else:
    if any(p.exists() for p in paths.values()):
        old = P['subsets'] / 'superseded'
        old.mkdir(exist_ok=True)
        for p in paths.values():
            if p.exists():
                shutil.move(str(p), str(old / f'{p.stem}_v{current_version(p)}.csv'))
        log('STAR sets from an older notebook version moved to data/subsets/superseded/ and re-sampled '
            '(no results had been produced from them)', 'WARN')
    seed = CFG['seed'] + 10
    margin = SC['sham_margin_s']
    seq_ok = SQ[(SQ.type == 'Sequence') & (SQ.status == 'ok') & (SQ.answer_idx >= 0) & (SQ.answer_match != 'order_only')]
    inter = SQ[(SQ.type == 'Interaction') & (SQ.answer_idx >= 0)]
    seq_pick = take_capped(seq_ok, SC['n_sequence'], SC['max_q_per_video'], seed)
    used = seq_pick.video_id.value_counts()
    inter = inter.assign(_room=SC['max_q_per_video'] - inter.video_id.map(used).fillna(0).astype(int))
    inter = inter[inter._room > 0].sample(frac=1, random_state=seed + 1)
    inter = inter[inter.groupby('video_id').cumcount() < inter._room].drop(columns='_room')
    int_pick = inter.head(SC['n_interaction']).reset_index(drop=True)
    main = pd.concat([seq_pick, int_pick], ignore_index=True)

    def sham_slot(r):
        # STAR clips are cut tightly around the actions, so there is rarely room before the first action
        # or after the second one ends. The onset gap only depends on the two START times, so a sham pause
        # anywhere before the first start or after the second start leaves it unchanged. Like the real
        # pause, it may fall inside an action; the only difference is whether it lies between the onsets.
        dur = r.clip_end - r.clip_start
        s1, s2 = r[f'{r.first_role}_start'], r[f'{r.second_role}_start']
        before, after = s1, dur - s2
        if before >= 2 * margin and before >= after:
            return round(before / 2, 2)
        if after >= 2 * margin:
            return round(s2 + after / 2, 2)
        if before >= 2 * margin:
            return round(before / 2, 2)
        return None
    cand = seq_pick[seq_pick.onset_gap_s > 0].copy()
    cand['pause_at_s'] = cand.apply(lambda r: round(r[f'{r.first_role}_start'] + r.onset_gap_s / 2, 2), axis=1)
    cand['sham_at_s'] = cand.apply(sham_slot, axis=1)
    print(f'gap candidates: {len(cand)} with onset gap > 0, {int(cand.sham_at_s.notna().sum())} with room for a sham pause')
    gapset =cand.dropna(subset=['sham_at_s']).sample(frac=1, random_state=seed + 2).head(SC['n_gap']).reset_index(drop=True)

    STAR_SETS = {'star_main': main.assign(set_version=STAR_SET_VERSION),
                 'star_gap': gapset.assign(set_version=STAR_SET_VERSION)}
    for n, d in STAR_SETS.items():
        d.to_csv(paths[n], index=False)
    log('STAR sets sampled and frozen')

for n, d in STAR_SETS.items():
    print(f'{n:10s} {len(d):4d} questions, {d.video_id.nunique():4d} source videos, types {d.type.value_counts().to_dict()}')
    if n == 'star_main' and len(d):
        sq = d[d.type == 'Sequence']
        for col, label in [('gap_s', 'end-to-start gap'), ('onset_gap_s', 'onset gap')]:
            g = sq[col]
            print(f'           {label} (s): median {g.median():.2f}, 90th pct {g.quantile(0.9):.2f}, max {g.max():.2f}')
planned = {'star_main': SC['n_sequence'] + SC['n_interaction'], 'star_gap': SC['n_gap']}
short = {n: f'{len(STAR_SETS[n])} of {k}' for n, k in planned.items() if len(STAR_SETS[n]) < k}
if short:
    log(f'fewer STAR questions than planned: {short}. Report the actual counts in the paper.', 'WARN')
assert len(STAR_SETS['star_gap']) > 0, 'star_gap is empty: send this output to fix the sham-slot rule'
assert set(STAR_SETS['star_gap'].uid) <= set(STAR_SETS['star_main'].uid)

[09:31:49] WARN  STAR sets from an older notebook version moved to data/subsets/superseded/ and re-sampled (no results had been produced from them)
gap candidates: 240 with onset gap > 0, 240 with room for a sham pause
[09:31:49] INFO  STAR sets sampled and frozen
star_main   360 questions,  281 source videos, types {'Sequence': 240, 'Interaction': 120}
           end-to-start gap (s): median -2.00, 90th pct 8.42, max 23.00
           onset gap (s): median 6.45, 90th pct 18.01, max 47.00
star_gap    120 questions,  109 source videos, types {'Sequence': 120}


## 5. Clips, then pause and sham versions

Each needed Charades video is extracted from the 480p zip, cut to its STAR clip, and re-encoded. For each `star_gap` question, four more versions are written:

| Version | What is inserted | Where |
| --- | --- | --- |
| `orig` | nothing (re-encoded the same way as the others) | — |
| `pause10`, `pause25` | a frozen frame lasting 10 s or 25 s | midway between the two key actions' starts |
| `sham10`, `sham25` | the same frozen frame length | outside the two key actions |

All versions pass through the same encoder, so any encoding artefact is identical. The manifest stores each version's shifted action times, so evidence scoring stays exact.

In [6]:
import cv2

def write_version(src_frames, fps, size, out_path, insert_at_s=None, pause_s=0.0):
    w = cv2.VideoWriter(str(out_path), cv2.VideoWriter_fourcc(*'mp4v'), fps, size)
    insert_idx = None if insert_at_s is None else int(round(insert_at_s * fps))
    for i, fr in enumerate(src_frames):
        if insert_idx is not None and i == insert_idx:
            for _ in range(int(round(pause_s * fps))):
                w.write(fr)
        w.write(fr)
    if insert_idx is not None and insert_idx >= len(src_frames):     # insertion at the very end
        for _ in range(int(round(pause_s * fps))):
            w.write(src_frames[-1])
    w.release()

def read_clip(path, start, end):
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    cap.set(cv2.CAP_PROP_POS_MSEC, max(0.0, start) * 1000)
    frames = []
    while True:
        ok, fr = cap.read()
        if not ok or cap.get(cv2.CAP_PROP_POS_MSEC) / 1000 > end + 1e-3:
            break
        frames.append(fr)
    cap.release()
    if not frames:
        raise ValueError(f'no frames read from {path.name} between {start}s and {end}s')
    h, w = frames[0].shape[:2]
    return frames, fps, (w, h)

CLIPS = P['local_videos'] / 'star'
CLIPS.mkdir(parents=True, exist_ok=True)
manifest_path = STAR / 'clip_manifest.csv'
main, gapset = STAR_SETS['star_main'], STAR_SETS['star_gap']
gap_uids = set(gapset.uid)
zip_on_drive = P['data'] / 'videos_star_eval.zip'
expected_clips = len(main) + 4 * len(gapset)

def manifest_matches():
    if not (zip_on_drive.exists() and manifest_path.exists()):
        return False
    m = pd.read_csv(manifest_path)
    want = {(u, 'orig'): None for u in main.uid}
    for g in gapset.itertuples(index=False):
        for v in ('pause10', 'pause25'):
            want[(g.uid, v)] = round(float(g.pause_at_s), 2)
        for v in ('sham10', 'sham25'):
            want[(g.uid, v)] = round(float(g.sham_at_s), 2)
    have = {(u, v): (None if pd.isna(a) else round(float(a), 2)) for u, v, a in zip(m.uid, m.version, m.insert_at_s)}
    return have == want
if manifest_matches():
    log('videos_star_eval.zip matches the current STAR sets; nothing to rebuild')
else:
    if manifest_path.exists():
        log('clip manifest does not match the current STAR sets: rebuilding all clips', 'WARN')
        shutil.rmtree(CLIPS, ignore_errors=True); CLIPS.mkdir(parents=True, exist_ok=True)
    src_zip = fetch('Charades_v1_480.zip')
    SRC = P['local_tmp'] / 'charades_src'
    SRC.mkdir(exist_ok=True)
    with zipfile.ZipFile(src_zip) as zf:
        index = {Path(n).stem: n for n in zf.namelist() if n.lower().endswith('.mp4')}
        for vid in sorted(set(main.video_id)):
            out = SRC / f'{vid}.mp4'
            if not out.exists() and vid in index:
                with zf.open(index[vid]) as s, open(out, 'wb') as d:
                    shutil.copyfileobj(s, d)
    missing_src = sorted(set(main.video_id) - {p.stem for p in SRC.glob('*.mp4')})
    if missing_src:
        log(f'{len(missing_src)} STAR source videos not in the Charades zip: {missing_src[:10]}', 'WARN')

    rows = []
    def make_clips(r):
        src = SRC / f'{r.video_id}.mp4'
        frames, fps, size = read_clip(src, r.clip_start, r.clip_end)
        base = r.uid
        versions = [('orig', None, 0.0)]
        if r.uid in gap_uids:
            g = gapset.set_index('uid').loc[r.uid]
            for secs, tag in [(SC['pause_medium_s'], '10'), (SC['pause_long_s'], '25')]:
                versions += [(f'pause{tag}', float(g.pause_at_s), secs), (f'sham{tag}', float(g.sham_at_s), secs)]
        for ver, at, secs in versions:
            out = CLIPS / f'{base}__{ver}.mp4'
            if not out.exists():
                write_version(frames, fps, size, out, at, secs)
            row = {'clip_id': out.stem, 'uid': r.uid, 'version': ver, 'insert_at_s': at, 'pause_s': secs,
                   'fps': round(fps, 3), 'n_frames_src': len(frames)}
            if r.type == 'Sequence' and pd.notna(getattr(r, 'anchor_start', None)):
                for role in ('anchor', 'answer'):
                    for edge in ('start', 'end'):
                        t = float(getattr(r, f'{role}_{edge}'))
                        row[f'{role}_{edge}'] = round(t + (secs if at is not None and t >= at else 0.0), 2)
                row['gap_s'] = round(row[f'{r.second_role}_start'] - row[f'{r.first_role}_end'], 2)
                row['onset_gap_s'] = round(row[f'{r.second_role}_start'] - row[f'{r.first_role}_start'], 2)
            rows.append(row)
    res = run_resumable([r for r in main.itertuples(index=False) if r.video_id not in missing_src],
                        key=lambda r: r.uid, fn=make_clips, desc='STAR clips', progress_every=50)
    print(res)
    M = pd.DataFrame(rows)
    M.to_csv(manifest_path, index=False)
    tmp = P['local_tmp'] / zip_on_drive.name
    with zipfile.ZipFile(tmp, 'w', zipfile.ZIP_STORED) as z:
        for cid in M.clip_id:
            z.write(CLIPS / f'{cid}.mp4', f'star/{cid}.mp4')
    shutil.copy2(tmp, zip_on_drive)
    log(f'videos_star_eval.zip: {len(M)} clips, {_gb(zip_on_drive.stat().st_size)} GB on Drive')

M = pd.read_csv(manifest_path)
print(M.version.value_counts().to_dict())
if 'onset_gap_s' in M:
    g = M[M.uid.isin(gap_uids)].groupby('version')
    print('median onset gap by version (s):', g.onset_gap_s.median().round(2).to_dict())
    print('median end-to-start gap by version (s):', g.gap_s.median().round(2).to_dict())

[09:31:50] WARN  clip manifest does not match the current STAR sets: rebuilding all clips
[09:31:50] INFO  downloading Charades_v1_480.zip from https://ai2-public-datasets.s3-us-west-2.amazonaws.com/charades/Charades_v1_480.zip
[09:40:13] INFO  Charades_v1_480.zip: 15.22 GB
[09:40:19] INFO  STAR clips: 360 total, 0 already done, 360 to do
[09:41:50] INFO  STAR clips: 50/360 | 1.8s each | ETA 9.4 min
[09:43:28] INFO  STAR clips: 100/360 | 1.9s each | ETA 8.2 min
[09:45:16] INFO  STAR clips: 150/360 | 2.0s each | ETA 6.9 min
[09:47:20] INFO  STAR clips: 200/360 | 2.1s each | ETA 5.6 min
[09:48:49] INFO  STAR clips: 250/360 | 2.0s each | ETA 3.7 min
[09:49:07] INFO  STAR clips: 300/360 | 1.8s each | ETA 1.8 min
[09:49:25] INFO  STAR clips: 350/360 | 1.6s each | ETA 0.3 min
{'total': 360, 'skipped': 0, 'ok': 360, 'failed': 0, 'remaining': 0, 'seconds': 548.5}
[09:49:57] INFO  videos_star_eval.zip: 840 clips, 1.57 GB on Drive
{'orig': 360, 'pause10': 120, 'sham10': 120, 'pause25': 120, 'sha

## 6. Ground-truth boxes for detection metrics

The Action Genome person and object boxes on STAR keyframes become per-video JSON files. Each frame lists its boxes, with class names and times in seconds within the clip. Experiment E1 scores detection against these. Action Genome has no track IDs, so tracking metrics still use VidOR on the dev videos.

In [7]:
obj_pkl = fetch('object_bbox_and_relationship.pkl', required=False)
per_pkl = fetch('person_bbox.pkl', required=False)
if obj_pkl and per_pkl:
    with open(obj_pkl, 'rb') as f:
        OBJB = pickle.load(f)
    with open(per_pkl, 'rb') as f:
        PERB = pickle.load(f)
    clip_of = main.groupby('video_id').agg(start=('clip_start', 'min'), end=('clip_end', 'max')).to_dict('index')
    wanted = set(clip_of)
    per_video = {}
    for key, objs in OBJB.items():                       # key like 'BE43L.mp4/000132.png'
        vid = key.split('.mp4')[0]
        if vid not in wanted:
            continue
        fid = int(Path(key).stem); fps = FPS.get(vid + '.mp4') or 30.0
        t = fid / fps - clip_of[vid]['start']
        boxes = []
        for o in objs or []:
            bb = o.get('bbox')
            if bb is not None and o.get('visible', True):
                x, y, w, h = [float(v) for v in bb]
                boxes.append({'cls': o.get('class'), 'xyxy': [x, y, x + w, y + h]})
        pb = PERB.get(key, {})
        for bb in np.asarray(pb.get('bbox', np.zeros((0, 4)))).reshape(-1, 4):
            boxes.append({'cls': 'person', 'xyxy': [float(v) for v in bb]})
        per_video.setdefault(vid, []).append({'frame': fid, 't': round(t, 3), 'boxes': boxes})
    for vid, frames in per_video.items():
        save_json(sorted(frames, key=lambda d: d['frame']), STAR / 'boxes' / f'{vid}.json')
    del OBJB, PERB; free_memory()
    print(f'ground-truth boxes saved for {len(per_video)}/{len(wanted)} source videos')
else:
    log('Action Genome box files unavailable; E1 detection metrics will use VidOR dev videos only.', 'WARN')

[09:49:57] INFO  downloading object_bbox_and_relationship.pkl from https://star-benchmark.s3.us-east.cloud-object-storage.appdomain.cloud/Annotations/object_bbox_and_relationship.pkl
[09:50:00] INFO  object_bbox_and_relationship.pkl: 0.13 GB
[09:50:00] INFO  downloading person_bbox.pkl from https://star-benchmark.s3.us-east.cloud-object-storage.appdomain.cloud/Annotations/person_bbox.pkl
[09:50:03] INFO  person_bbox.pkl: 0.15 GB
ground-truth boxes saved for 281/281 source videos


## 7. Oracle memory inputs

For every `star_main` question, the ground truth is saved in clip time:
- all Charades actions overlapping the clip, as events with start and end,
- the objects seen in each keyframe,
- the relationships in each keyframe.

Notebook 04 turns these into oracle versions of M1–M4, so the "perfect perception" runs use the same memory formats as the real ones. For pause and sham versions, times are shifted using the manifest.

In [8]:
CHA_NAME = CHA_CLASSES
def oracle_record(q, r):
    vid, cs, ce = q['video_id'], float(q['start']), float(q['end'])
    events = [{'event': CHA_NAME.get(c, c), 'class': c, 'start': round(max(s, cs) - cs, 2),
               'end': round(min(e, ce) - cs, 2)}
              for c, s, e in CHA_ACTIONS.get(vid, []) if e > cs and s < ce]
    fps = FPS.get(vid + '.mp4') or 30.0
    frames = []
    for f in sorted(q['situations'], key=int):
        s = q['situations'][f]
        frames.append({'t': round(int(f) / fps - cs, 2),
                       'objects': [OBJ.get(o, o) for o in s.get('bbox_labels', [])],
                       'relations': [[OBJ.get(a, a), REL.get(l, l), OBJ.get(b, b)]
                                     for (a, b), l in zip(s.get('rel_pairs', []), s.get('rel_labels', []))],
                       'actions': [ACTION_NAME.get(a, a) for a in s.get('actions', [])]})
    return {'uid': r.uid, 'video_id': vid, 'duration_s': round(ce - cs, 2),
            'events': sorted(events, key=lambda e: e['start']), 'keyframes': frames}

by_qid = {q['question_id']: q for q in SV}
for f in (STAR / 'oracle').glob('*.json'):            # drop oracle files for questions no longer in the sets
    if f.stem not in set(main.uid):
        f.unlink()
n = 0
for r in main.itertuples(index=False):
    out = STAR / 'oracle' / f'{r.uid}.json'
    if not out.exists():
        save_json(oracle_record(by_qid[r.question_id], r), out); n += 1
print(f'oracle inputs: {n} new, {len(list((STAR / "oracle").glob("*.json")))} total')
del SV; free_memory()

oracle inputs: 340 new, 360 total


## 8. Record

In [9]:
card = load_json(P['subsets'] / 'data_card.json', default={})
card['star'] = {
    'created': dt.datetime.now().isoformat(timespec='seconds'),
    'sets': {n: {'questions': len(d), 'videos': int(d.video_id.nunique()), 'types': d.type.value_counts().to_dict()}
             for n, d in STAR_SETS.items()},
    'set_hashes': {n: hashlib.sha1(d.uid.astype(str).str.cat(sep='|').encode()).hexdigest()[:12] for n, d in STAR_SETS.items()},
    'clips': int(len(M)), 'zip_gb': _gb((P['data'] / 'videos_star_eval.zip').stat().st_size) if (P['data'] / 'videos_star_eval.zip').exists() else None,
    'natural_gap_median_s': float(STAR_SETS['star_main'].gap_s.median()) if 'gap_s' in STAR_SETS['star_main'] else None,
}
save_json(card, P['subsets'] / 'data_card.json')
RUN.finish('done', notes=json.dumps(card['star']['sets']))
print(json.dumps(card['star'], indent=1))

[09:51:01] INFO  run finished: 01b_star-20261016-093129-4239b5 [done] 19.5 min, peak RAM 2.49 GB, peak VRAM 0.0 GB
{
 "created": "2026-10-16T09:51:00",
 "sets": {
  "star_main": {
   "questions": 360,
   "videos": 281,
   "types": {
    "Sequence": 240,
    "Interaction": 120
   }
  },
  "star_gap": {
   "questions": 120,
   "videos": 109,
   "types": {
    "Sequence": 120
   }
  }
 },
 "set_hashes": {
  "star_main": "deb5b86a9747",
  "star_gap": "3df7b5466808"
 },
 "clips": 840,
 "zip_gb": 1.57,
 "natural_gap_median_s": -2.0
}
